In [1]:
from sentence_transformers import SentenceTransformer

# Use a public model that does not require Hugging Face authentication
model = SentenceTransformer("all-MiniLM-L6-v2")

In [3]:
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]
embeddings = model.encode(sentences)
print(embeddings.shape)

In [4]:
similarities = model.similarity(embeddings, embeddings)
print(similarities)

Run it in the Google Collab

In [1]:
!pip install -U sentence-transformers
!pip install -U transformers

### GysBERT Based

In [1]:
!ls

extracted_entities_20.csv  sample_data


In [ ]:
# read csv as pandas dataframe
import pandas as pd

df = pd.read_csv("extracted_entities_20.csv")
all_PERSON = df['person'].tolist()

In [2]:
# Sentences we want sentence embeddings for
sentences = [
    "Het weer is vandaag heerlijk.",
    "Het is buiten ongelooflijk zonnig.",
    "De kat slaapt op de mat."
]

In [3]:
sentences = all_PERSON

In [ ]:
!git clone https://huggingface.co/emanjavacas/GysBERT

Cloning into 'GysBERT'...
remote: Enumerating objects: 17, done.
remote: Total 17 (delta 0), reused 0 (delta 0), pack-reused 17 (from 1)
Receiving objects: 100% (17/17), 104.40 KiB | 6.52 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [5]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load model from HuggingFace Hub
model_name = "emanjavacas/GysBERT"
model_name = "GysBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, dtype="auto")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GysBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling. In this case, max pooling.
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

print("Sentence embeddings:")
print(sentence_embeddings)

# calculate cosine similarity between the two sentence embeddings
cosine_similarity = torch.nn.functional.cosine_similarity(sentence_embeddings[0], sentence_embeddings[1], dim=0)
print("Cosine similarity between the two sentences:", cosine_similarity.item())

# report grid similarity between all sentence embeddings
# compute full pairwise cosine similarity matrix for all sentence embeddings
normalized = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)
grid_similarity = torch.matmul(normalized, normalized.T)
print("Grid similarity between all sentence embeddings:")
print(grid_similarity)

NameError: name 'AutoTokenizer' is not defined

In [17]:
from sentence_transformers import util

similarity_matrix = util.cos_sim(sentence_embeddings, sentence_embeddings)
similarity_matrix

In [ ]:
# given the similarity matrix, find the most similar sentences for each sentence when the similarity is above threshold
threshold = 0.7
most_similar_sentences = {}
for i in range(len(sentences)):
    # for i give me integer index of j when similarity score is > threshold
    similar_indices = torch.where(similarity_matrix[i] > threshold)[0]
    if len(similar_indices) > 0:
        most_similar_index = similar_indices[torch.argmax(similarity_matrix[i][similar_indices])]
        # get the similarity score
        similarity_score = similarity_matrix[i][most_similar_index].item()
        most_similar_sentences[sentences[i]] = (sentences[most_similar_index], similarity_score)
